In [ ]:
# Referência: Mohammadi
import pandas as pd
import os
import numpy as np
import spacy
import re
import ast # Para converter a string de entrada em um objeto Python de forma segura
from spacy.lang.pt.stop_words import STOP_WORDS
from collections import Counter

# Carrega o modelo de linguagem portuguesa da spaCy
nlp = spacy.load("pt_core_news_sm")

In [ ]:
LINKING_VERBS = {
    "ser",
    "estar",
    "parecer",
    "permanecer",
    "continuar",
    "ficar",
    "andar",
    "tornar-se",
    "resultar",
    "revelar-se",
    "mostrar-se",
    "aparecer"
}

IGNORE_WORDS = {
    "patentes",
}


In [ ]:
df_novo = pd.read_csv('./datasets/database-filtrado.csv.zip', compression='zip')

In [ ]:
def clean_and_preprocess_text(text: str) -> list[str]:
    """
    Executa um pipeline completo de pré-processamento em um texto em português.

    Args:
        text: O texto original a ser processado.

    Returns:
        Uma lista de tokens lematizados, sem stop words, pontuação ou números.
    """
    # 1. Normalização inicial: remove URLs, múltiplos espaços e deixa em minúsculas
    # Regex para remover URLs que possam existir
    text = re.sub(r'\((?:\s*[\dA-Za-z]{1,4}(?:\s*,\s*[\dA-Za-z]{1,4})*\s*)\)', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Regex para remover padrões como (A), (A1, B2, C) ou (1, 2, 3)
    text = re.sub(r'\(\s*\w{1,4}(?:\s*,\s*\w{1,4})*\s*\)', '', text)

    # Regex para remover caracteres que não sejam letras ou espaços
    text = re.sub(r'[^a-zA-ZáàâãéèêíïóôõöúçÁÀÂÃÉÈÊÍÏÓÔÕÖÚÇ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    # 2. Processamento com spaCy
    # A spaCy processa o texto para criar um objeto 'doc' com todas as análises
    doc = nlp(text)

    # 3. Remoção de Stop Words e Lematização
    # Usamos uma list comprehension para criar a lista final de tokens
    # - token.lemma_ : Retorna a forma base da palavra (lematização)
    # - not token.is_stop : Verifica se o token NÃO é uma stop word
    # - not token.is_punct : Verifica se o token NÃO é pontuação
    # - not in LINKING_VERBS : Remove verbos de ligação comuns
    # - lemma not in STOP_WORDS : Remove stop words personalizadas
    tokens = []
    for token in doc:
        lemma = token.lemma_.strip().lower()
        if (
            lemma
            and len(lemma) > 2
            and lemma not in STOP_WORDS
            and lemma not in LINKING_VERBS
            and not token.is_punct
            and " " not in lemma  # remove "em o", "de o", etc
        ):
            tokens.append(lemma)

    return tokens

def top_n_words(tokens: list[str], n: int = 30) -> list[tuple[str, int]]:
    return Counter(tokens).most_common(n)

In [ ]:
# ...existing code...
from concurrent.futures import ThreadPoolExecutor
import os

# df_novo.iloc[15].to_csv("example.csv")

df_novo['title_abstract'] = df_novo['invention_title_text'] + " " + df_novo['abstract_text']

# Processamento em multithread
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    df_novo['tokens'] = list(executor.map(clean_and_preprocess_text, df_novo['title_abstract']))

#processed_tokens = clean_and_preprocess_text(df_novo['title_abstract'][2412])
#print(processed_tokens)
# ...existing code...

In [ ]:
# ...existing code...
all_tokens = []
for token_list in df_novo['tokens']:
    all_tokens.extend(token_list)
# Conta e ordena por frequência
word_freq = Counter(all_tokens).most_common()

df_freq = pd.DataFrame(word_freq, columns=['palavra', 'frequencia'])

# Adiciona coluna com tamanho da string e ordena por ela
df_freq['tamanho'] = df_freq['palavra'].str.len()
df_freq = df_freq.sort_values('tamanho', ascending=False)

# Exporta para CSV
df_freq.to_csv('./datasets/palavras_frequencia.csv', index=False)

print("Top 20 palavras mais frequentes:")
for word, freq in word_freq[:20]:
    print(f"{word}: {freq}")

print(f"\nArquivo salvo em: ./datasets/palavras_frequencia.csv")
print(f"Total de palavras únicas: {len(df_freq)}")

# Mostra as 20 palavras mais longas
print("\nTop 20 palavras mais longas:")
print(df_freq[['palavra', 'tamanho']].head(20))
# ...existing code...

In [ ]:
df_novo.to_csv('./datasets/database-lemmetizado.csv.zip', index=False, compression='zip')